# Laboratorio 3 - Evaluar entidades

Individual - 2.5% - 6 de octubre de 2026, 23:59 Monterrey.

## Preparación

Abre este notebook dentro del paquete extraído. La primera celda encuentra `entidades.json`; no necesitas copiar rutas personales. Usa el entorno del curso (Python 3.11, CPU y perfil `requirements/linguistica.txt`). Si ya funciona, no reinstales.

In [1]:
from pathlib import Path
import json, platform, re, sys
from importlib.metadata import version
import pandas as pd
import spacy
from IPython.display import display
# Busca desde la carpeta del notebook hacia arriba, sin rutas personales.
candidatos = [Path.cwd(), *Path.cwd().parents]
rutas = [p / sub for p in candidatos for sub in
         ['data/entidades.json', 'entidades.json']]
DATOS = next((p for p in rutas if p.is_file()), None)
if DATOS is None:
    raise FileNotFoundError('Extrae el paquete completo y abre el notebook dentro de él. Falta entidades.json.')
registros = json.loads(DATOS.read_text(encoding='utf-8'))
versiones = {'Python': platform.python_version(), 'spaCy': version('spacy'), 'pandas': version('pandas')}
print(versiones, '\nDatos:', DATOS, '\nRegistros:', len(registros))

{'Python': '3.14.4', 'spaCy': '3.8.16', 'pandas': '3.0.5'} 
Datos: C:\Users\oscar\Desktop\AI_concentracion\Clase05_Lab3\build\entidades.json 
Registros: 48


## Preparar y leer

**Predice:** «Lía usa DemoApp.» tiene DemoApp en `[8:15]`. Si la propuesta termina en 14, ¿cuántos TP, FP y FN habrá? Este ejemplo está resuelto para mostrar el mecanismo; no pertenece a tu test.

`&` encuentra las tuplas compartidas por ambos conjuntos. `A - B` conserva las que están en A y no en B.

In [2]:
texto="Lía usa DemoApp."
referencia={("demo",8,15,"PRODUCTO")}
propuesta={("demo",8,14,"PRODUCTO")}
print("Referencia:",texto[8:15],"Propuesta:",texto[8:14])
aciertos=referencia & propuesta
sobrantes=propuesta - referencia
faltantes=referencia - propuesta
print({"TP":len(aciertos),"FP":len(sobrantes),"FN":len(faltantes)})
assert (len(aciertos),len(sobrantes),len(faltantes))==(0,1,1)

Referencia: DemoApp Propuesta: DemoAp
{'TP': 0, 'FP': 1, 'FN': 1}


**Compara:** el corte incorrecto pierde la última letra. No coincide: TP=0, FP=1, FN=1. Un solo límite incorrecto deja una propuesta sobrante y una referencia faltante. Ahora aplica esa idea a tu muestra.

In [3]:
muestra=[r for r in registros if r['split']=='test'][:5]
assert len(muestra)==5 and len({r['id'] for r in muestra})==5
display(pd.DataFrame([{'id':r['id'],'texto':r['text'],'anotaciones':r['entities']} for r in muestra]))

,id,texto,anotaciones
0,E10_0,La aplicación MesaApp falla; el folio es INC-1...,"[[14, 21, PRODUCTO], [41, 49, FOLIO]]"
1,E10_1,Favor de revisar INC-1011: no abre MesaApp.,"[[17, 25, FOLIO], [35, 42, PRODUCTO]]"
2,E10_2,Con MesaApp no puedo guardar. Seguimiento: INC...,"[[4, 11, PRODUCTO], [43, 51, FOLIO]]"
3,E10_3,Mi solicitud INC-1011 corresponde a MesaApp.,"[[13, 21, FOLIO], [36, 43, PRODUCTO]]"
4,E11_0,La aplicación NubeTec falla; el folio es INC-1...,"[[14, 21, PRODUCTO], [41, 49, FOLIO]]"


## Extraer

El catálogo trabajado es AulaPro, MesaApp y NubeTec; el folio empieza con INC- y tiene cuatro dígitos. Declara estas reglas antes de medir. No adaptes el catálogo a las respuestas del test.

Construye un procesador español y añade EntityRuler. Forma los conjuntos `gold` (referencia) y `pred` (predicciones) con tuplas `(id, inicio, fin, tipo)`. En `nlp(texto).ents`, cada entidad ofrece `start_char`, `end_char` y `label_`.

Antes de ejecutar, elige una predicción esperada y señala su fragmento. Después comprueba el corte sobre el texto original.

In [4]:
nlp = spacy.blank("es")
ruler = nlp.add_pipe("entity_ruler")
catalogo = ["AulaPro", "MesaApp", "NubeTec"]
reglas = [{"label": "PRODUCTO", "pattern": nombre} for nombre in catalogo]
reglas.append({"label": "FOLIO", "pattern": [{"TEXT": {"REGEX": r"^INC-\d{4}$"}}]})
ruler.add_patterns(reglas)

textos = {r["id"]: r["text"] for r in muestra}
gold = {(r["id"], a, b, tipo) for r in muestra for a, b, tipo in r["entities"]}
pred = set()
for r in muestra:
    for ent in nlp(r["text"]).ents:
        assert r["text"][ent.start_char:ent.end_char] == ent.text
        pred.add((r["id"], ent.start_char, ent.end_char, ent.label_))

tabla_pred = pd.DataFrame(sorted(pred), columns=["id", "inicio", "fin", "tipo"])
tabla_pred["fragmento"] = [textos[i][a:b] for i, a, b, _ in sorted(pred)]
display(tabla_pred)
print("referencias:", len(gold), "| predicciones:", len(pred))
ejemplo = ("E10_1", 17, 25, "FOLIO")
print(ejemplo in pred, repr(textos["E10_1"][17:25]), "| siguiente carácter:", repr(textos["E10_1"][25]))

,id,inicio,fin,tipo,fragmento
0,E10_0,14,21,PRODUCTO,MesaApp
1,E10_0,41,49,FOLIO,INC-1011
2,E10_1,17,25,FOLIO,INC-1011
3,E10_1,35,42,PRODUCTO,MesaApp
4,E10_2,4,11,PRODUCTO,MesaApp
5,E10_2,43,51,FOLIO,INC-1011
6,E10_3,13,21,FOLIO,INC-1011
7,E10_3,36,43,PRODUCTO,MesaApp
8,E11_0,14,21,PRODUCTO,NubeTec
9,E11_0,41,49,FOLIO,INC-1012


referencias: 10 | predicciones: 10
True 'INC-1011' | siguiente carácter: ':'


## Contar y medir

Predice qué pasaría con FP y FN si una entidad tuviera el tipo equivocado. Luego calcula los conteos originales usando intersección y diferencias.

Define `TP`, `FP`, `FN`, `precision`, `recall` y `f1`. Las fórmulas están en el enunciado. Declara tu convenio para denominadores cero. Comprueba TP+FN frente al número de referencias y TP+FP frente al número de predicciones.

Explica una división sustituyendo tus números: ¿qué representa el numerador y qué estás contando en el denominador?

In [5]:
aciertos = gold & pred
TP = len(aciertos)
FP = len(pred - gold)
FN = len(gold - pred)
precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print({"TP": TP, "FP": FP, "FN": FN})
print(f"precision = TP/(TP+FP) = {TP}/{TP + FP} = {precision}")
print(f"recall = TP/(TP+FN) = {TP}/{TP + FN} = {recall}")
print(f"F1 = 2*{precision}*{recall}/({precision}+{recall}) = {f1}")
assert TP + FN == len(gold)
assert TP + FP == len(pred)

{'TP': 10, 'FP': 0, 'FN': 0}
precision = TP/(TP+FP) = 10/10 = 1.0
recall = TP/(TP+FN) = 10/10 = 1.0
F1 = 2*1.0*1.0/(1.0+1.0) = 1.0


## Diagnosticar

Analiza dos errores de límite o tipo y propón una corrección. Si la muestra no los produce, simúlalos en una **copia** de las predicciones. Anticipa qué conteos cambiarán antes de ejecutar.

Muestra el fragmento y el tipo esperado junto a lo propuesto. Indica si el error fue observado o simulado. Conserva las métricas originales separadas; no incluyas los mensajes del desafío en parejas.

In [6]:
def metricas(referencia, propuesta):
    tp = len(referencia & propuesta)
    fp = len(propuesta - referencia)
    fn = len(referencia - propuesta)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "precision": p, "recall": r, "F1": f}

def valida(tupla):
    i, a, b, tipo = tupla
    fragmento = textos[i][a:b]
    if tipo == "PRODUCTO":
        return fragmento in catalogo
    return re.fullmatch(r"INC-\d{4}", fragmento) is not None

original = metricas(gold, pred)
esperado_limite = ("E10_0", 41, 49, "FOLIO")
esperado_tipo = ("E11_0", 14, 21, "PRODUCTO")
assert esperado_limite in pred and esperado_tipo in pred

error_limite = ("E10_0", 41, 48, "FOLIO")
error_tipo = ("E11_0", 14, 21, "FOLIO")
filas = []
for nombre, bueno, malo in [("límite", esperado_limite, error_limite), ("tipo", esperado_tipo, error_tipo)]:
    copia = (pred - {bueno}) | {malo}
    m = metricas(gold, copia)
    filtrada = metricas(gold, {t for t in copia if valida(t)})
    assert (m["TP"], m["FP"], m["FN"]) == (original["TP"] - 1, original["FP"] + 1, original["FN"] + 1)
    assert (filtrada["TP"], filtrada["FP"], filtrada["FN"]) == (original["TP"] - 1, 0, original["FN"] + 1)
    filas.append({"error": nombre, "origen": "simulado", "id": bueno[0],
                  "esperado": f"{textos[bueno[0]][bueno[1]:bueno[2]]!r} {bueno[3]}",
                  "propuesto": f"{textos[malo[0]][malo[1]:malo[2]]!r} {malo[3]}",
                  "TP": m["TP"], "FP": m["FP"], "FN": m["FN"], "precision": m["precision"],
                  "recall": m["recall"], "F1": m["F1"], "TP/FP/FN tras valida": f"{filtrada['TP']}/{filtrada['FP']}/{filtrada['FN']}"})
display(pd.DataFrame(filas))
print("métricas originales, sin tocar:", {"TP": TP, "FP": FP, "FN": FN, "precision": precision, "recall": recall, "F1": f1})
assert all(valida(t) for t in pred) and pred == pred - {error_limite, error_tipo}

,error,origen,id,esperado,propuesto,TP,FP,FN,precision,recall,F1,TP/FP/FN tras valida
0,límite,simulado,E10_0,'INC-1011' FOLIO,'INC-101' FOLIO,9,1,1,0.9,0.9,0.9,9/0/1
1,tipo,simulado,E11_0,'NubeTec' PRODUCTO,'NubeTec' FOLIO,9,1,1,0.9,0.9,0.9,9/0/1


métricas originales, sin tocar: {'TP': 10, 'FP': 0, 'FN': 0, 'precision': 1.0, 'recall': 1.0, 'F1': 1.0}


## Mi explicación

1. Elegí ("E10_1", 17, 25, "FOLIO"). texto[17:25] da "INC-1011": son 8 caracteres, desde la posición 17 hasta la 24. El fin 25 queda fuera y ahí está el ":" que sigue al folio, por eso la entidad termina en 25 y no en 24.

2. Mi precisión es 10/10 = 1.0: de las 10 entidades que propuse, las 10 coinciden exacto con una anotación. Mi recall es 10/10 = 1.0: de las 10 anotaciones, recuperé las 10. El F1 sale 1.0. TP+FN es 10, el total de referencias, y TP+FP es 10, el total de predicciones.

3. Mi extractor no falló en estos cinco textos, así que los dos errores son simulados y los hice sobre copias de pred. En el primero cambió el límite: en E10_0 acorté el folio a "INC-101" (fin 48 en vez de 49), y los conteos pasaron de 10/0/0 a TP 9, FP 1 y FN 1. En el segundo cambió el tipo: NubeTec de E11_0 quedó como FOLIO en vez de PRODUCTO, con los mismos conteos. En los dos, precisión, recall y F1 bajan a 0.9. Las métricas originales no se tocaron.

4. Propongo revisar cada propuesta contra el texto: un PRODUCTO debe estar en el catálogo y un FOLIO debe ser INC- con cuatro dígitos completos. Esa revisión es la función valida. Esperaba que descartara la propuesta mala y dejara FP en 0, con TP 9 y FN 1, y eso es lo que da la tabla. Para recuperar el TP hay que corregir la regla que produjo el error; en el folio, el patrón ^INC-\d{4}$ sobre el token ya hace que el span termine donde termina el folio.

5. Son cinco textos de dos familias (N10 y N11), con frases casi iguales, tres productos y un solo formato de folio. Un F1 de 1.0 aquí solo dice que mis reglas reproducen estas anotaciones. No me permite asegurar que reconozca productos nuevos, folios con otro formato o mensajes con errores de escritura, ni sé cómo fallaría una regla real, porque los dos errores que analicé los provoqué yo.

## Guardar

La siguiente celda organiza resultados que tú ya calculaste. Si falta trabajo, lo dice y no inventa salidas. Completa el README creado, guarda este notebook con tus resultados y entrega los archivos pertinentes. Puedes usar los nombres sugeridos para reconocer tus archivos. La rúbrica sigue siendo 25/35/30/10.

In [7]:
# Solo guarda lo que ya construiste; no calcula la solución.
SALIDA = Path.cwd() / 'salidas_laboratorio_03'
SALIDA.mkdir(exist_ok=True)
requeridas = ['pred','TP','FP','FN','precision','recall','f1']
if all(nombre in globals() for nombre in requeridas):
    pd.DataFrame(sorted(pred), columns=['id','inicio','fin','tipo']).to_csv(SALIDA/'predicciones.csv',index=False)
    resumen = dict(TP=TP,FP=FP,FN=FN,precision=precision,recall=recall,F1=f1)
    (SALIDA/'metricas.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
    readme = SALIDA/'README.md'
    if not readme.exists():
        readme.write_text('# Mi Laboratorio 3\n\nEjecutar el notebook desde arriba con kernel nuevo.\n'
                          + 'Versiones: '+json.dumps(versiones,ensure_ascii=False)
                          + '\n\nDatos: entidades.json. IDs: '+str([r['id'] for r in muestra])
                          + '\n\nDecisiones: COMPLETAR reglas, errores observados/simulados y límites.\n'
                          + 'El análisis de dos errores está en el notebook.\n',encoding='utf-8')
    print('Archivos guardados en:', SALIDA.resolve())
    print('Completa README.md y guarda el notebook desde Jupyter.')
else:
    print('Aún falta tu trabajo: completa las predicciones y métricas antes de exportar.')

Archivos guardados en: C:\Users\oscar\Desktop\AI_concentracion\Clase05_Lab3\build\salidas_laboratorio_03
Completa README.md y guarda el notebook desde Jupyter.


In [8]:
readme = SALIDA / "README.md"
lineas = [
    "# Laboratorio 3: evaluar entidades",
    "",
    "Orden de ejecución: abrir Laboratorio_03.ipynb dentro del paquete (junto a entidades.json), con kernel nuevo, y correr las celdas de arriba hacia abajo. La última celda de código crea esta carpeta.",
    "",
    "Versiones: " + json.dumps(versiones, ensure_ascii=False) + ". El curso pide Python 3.11; aquí corrió con la versión indicada.",
    "",
    "Datos: entidades.json, sin modificar. Muestra: los primeros cinco textos de test, " + ", ".join(r["id"] for r in muestra) + ".",
    "",
    "Reglas, definidas antes de medir: spacy.blank('es') con EntityRuler. PRODUCTO es AulaPro, MesaApp o NubeTec (frase exacta). FOLIO es un token que cumple ^INC-\\d{4}$. No hay aleatoriedad ni semilla.",
    "",
    "Evaluación: coincidencia exacta de (id, inicio, fin, tipo). Si un denominador es cero, la métrica vale 0.",
    "",
    f"Resultado en los cinco textos: TP={TP}, FP={FP}, FN={FN}, precisión={precision}, recall={recall}, F1={f1}.",
    "",
    "Errores: dos simulados sobre copias (límite en E10_0 y tipo en E11_0), cada uno con TP 9, FP 1, FN 1 y métricas de 0.9. El análisis está en el notebook.",
    "",
    "Límites: cinco textos de dos familias; un F1 alto aquí no prueba que reconozca productos o folios nuevos.",
    "",
    "Archivos: predicciones.csv, metricas.json y este README.",
]
readme.write_text("\n".join(lineas) + "\n", encoding="utf-8")
print(readme.read_text(encoding="utf-8"))

# Laboratorio 3: evaluar entidades

Orden de ejecución: abrir Laboratorio_03.ipynb dentro del paquete (junto a entidades.json), con kernel nuevo, y correr las celdas de arriba hacia abajo. La última celda de código crea esta carpeta.

Versiones: {"Python": "3.14.4", "spaCy": "3.8.16", "pandas": "3.0.5"}. El curso pide Python 3.11; aquí corrió con la versión indicada.

Datos: entidades.json, sin modificar. Muestra: los primeros cinco textos de test, E10_0, E10_1, E10_2, E10_3, E11_0.

Reglas, definidas antes de medir: spacy.blank('es') con EntityRuler. PRODUCTO es AulaPro, MesaApp o NubeTec (frase exacta). FOLIO es un token que cumple ^INC-\d{4}$. No hay aleatoriedad ni semilla.

Evaluación: coincidencia exacta de (id, inicio, fin, tipo). Si un denominador es cero, la métrica vale 0.

Resultado en los cinco textos: TP=10, FP=0, FN=0, precisión=1.0, recall=1.0, F1=1.0.

Errores: dos simulados sobre copias (límite en E10_0 y tipo en E11_0), cada uno con TP 9, FP 1, FN 1 y métricas de 0.9.

## Para consultar después

- [EntityRuler: reglas y entidades](https://spacy.io/api/entityruler).
- [Span: intervalos de texto y posiciones](https://spacy.io/api/span).
- [Análisis lingüístico](https://spacy.io/usage/linguistic-features).

Las demostraciones y los desafíos son formativos. Las métricas oficiales del Laboratorio 3 corresponden únicamente a sus cinco textos.